# CS601T Assignment 6 — RNN & LSTM Training (12 models)


In [1]:
import os
import json
import zipfile
import random
import csv
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Kaggle input data folder (uploaded Kaggle Dataset is mounted here).
# /kaggle/input/datasets/aditya000001/complete-dataset-for-assignment-6/dataset_prepared_centered/handwriting/train/a/1.npy
KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/assignment6_results")
WORK_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
CONVERGENCE_THRESHOLD = 1e-4
CONVERGENCE_CONSECUTIVE_EPOCHS = 3
MAX_EPOCHS = 2000  # Safety cap only; stopping is based on convergence.
NUM_WORKERS = 2

print("Results will be saved to:", WORK_DIR)


Device: cuda
Results will be saved to: /kaggle/working/assignment6_results


In [2]:
# Cell 2 — Locate/extract the prepared dataset

def locate_prepared_data():
    for candidate in KAGGLE_INPUT.rglob("handwriting"):
        if (candidate / "train").is_dir() and (candidate / "test").is_dir():
            cv_candidate = candidate.parent / "cv_speech"
            if (cv_candidate / "train").is_dir() and (cv_candidate / "test").is_dir():
                return candidate.parent

    zip_candidates = list(KAGGLE_INPUT.rglob("dataset_prepared_centered.zip"))
    if not zip_candidates:
        zip_candidates = [
            p for p in KAGGLE_INPUT.rglob("*.zip")
            if "prepared_centered" in p.name.lower()
        ]

    if zip_candidates:
        archive = zip_candidates[0]
        extract_dir = Path("/kaggle/working/dataset_extracted")
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive, "r") as zf:
            zf.extractall(extract_dir)

        for candidate in extract_dir.rglob("handwriting"):
            if (candidate / "train").is_dir() and (candidate / "test").is_dir():
                cv_candidate = candidate.parent / "cv_speech"
                if (cv_candidate / "train").is_dir() and (cv_candidate / "test").is_dir():
                    return candidate.parent

    raise FileNotFoundError(
        "Could not locate prepared data. Add dataset_prepared_centered.zip "
        "as a Kaggle Dataset input, or mount its extracted folders."
    )

DATA_ROOT = locate_prepared_data()
HANDWRITING_ROOT = DATA_ROOT / "handwriting"
CV_ROOT = DATA_ROOT / "cv_speech"

print("DATA_ROOT:", DATA_ROOT)
print("Handwriting train exists:", (HANDWRITING_ROOT / "train").exists())
print("CV train exists:", (CV_ROOT / "train").exists())

with open(HANDWRITING_ROOT / "class_mapping.json", "r") as f:
    HANDWRITING_CLASS_MAPPING = json.load(f)

with open(CV_ROOT / "class_mapping.json", "r") as f:
    CV_CLASS_MAPPING = json.load(f)

print("Handwriting classes:", HANDWRITING_CLASS_MAPPING)
print("CV classes:", CV_CLASS_MAPPING)


DATA_ROOT: /kaggle/input/datasets/aditya000001/complete-dataset-for-assignment-6/dataset_prepared_centered
Handwriting train exists: True
CV train exists: True
Handwriting classes: {'a': 0, 'ai': 1, 'chA': 2, 'lA': 3, 'tA': 4}
CV classes: {'hI': 0, 'ka': 1, 'rI': 2, 'sa': 3, 'tI': 4}


In [3]:
# Cell 3 — Load variable-length sequences and create train/validation/test sets

def load_split_sequences(dataset_root, split, class_mapping):
    sequences, labels, sample_ids = [], [], []
    split_dir = dataset_root / split

    for class_name, class_idx in sorted(class_mapping.items(), key=lambda kv: kv[1]):
        class_dir = split_dir / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Missing class directory: {class_dir}")

        for file_path in sorted(class_dir.glob("*.npy")):
            arr = np.load(file_path).astype(np.float32)

            if arr.ndim != 2:
                raise ValueError(f"Expected 2D sequence in {file_path}, got {arr.shape}")

            expected_features = 2 if dataset_root.name == "handwriting" else 39
            if arr.shape[1] != expected_features:
                raise ValueError(
                    f"Feature dimension mismatch in {file_path}: "
                    f"expected {expected_features}, got {arr.shape[1]}"
                )

            if len(arr) == 0:
                raise ValueError(f"Empty sequence: {file_path}")

            sequences.append(torch.tensor(arr, dtype=torch.float32))
            labels.append(int(class_idx))
            sample_ids.append(f"{class_name}/{file_path.name}")

    return sequences, np.asarray(labels, dtype=np.int64), sample_ids


class SequenceDataset(Dataset):
    def __init__(self, sequences, labels, sample_ids=None):
        self.sequences = sequences
        self.labels = torch.as_tensor(labels, dtype=torch.long)
        self.sample_ids = sample_ids or [str(i) for i in range(len(sequences))]

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx], self.sample_ids[idx]


def collate_sequences(batch):
    seqs, labels, sample_ids = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in seqs], dtype=torch.long)
    padded = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    labels = torch.stack(labels)
    return padded, lengths, labels, list(sample_ids)


def make_loaders(dataset_root, class_mapping, batch_size=BATCH_SIZE):
    train_sequences, train_labels, train_ids = load_split_sequences(
        dataset_root, "train", class_mapping
    )
    test_sequences, test_labels, test_ids = load_split_sequences(
        dataset_root, "test", class_mapping
    )

    indices = np.arange(len(train_labels))
    train_idx, val_idx = train_test_split(
        indices,
        test_size=0.20,
        random_state=SEED,
        stratify=train_labels
    )

    train_ds = SequenceDataset(
        [train_sequences[i] for i in train_idx],
        train_labels[train_idx],
        [train_ids[i] for i in train_idx]
    )
    val_ds = SequenceDataset(
        [train_sequences[i] for i in val_idx],
        train_labels[val_idx],
        [train_ids[i] for i in val_idx]
    )
    test_ds = SequenceDataset(test_sequences, test_labels, test_ids)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=collate_sequences, num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )
    train_eval_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=False,
        collate_fn=collate_sequences, num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        collate_fn=collate_sequences, num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        collate_fn=collate_sequences, num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

    print(
        f"{dataset_root.name}: total train={len(train_ds)+len(val_ds)}, "
        f"train split={len(train_ds)}, validation={len(val_ds)}, test={len(test_ds)}"
    )
    print("Train labels:", Counter(train_ds.labels.numpy()))
    print("Validation labels:", Counter(val_ds.labels.numpy()))
    print("Test labels:", Counter(test_ds.labels.numpy()))

    return train_loader, train_eval_loader, val_loader, test_loader


In [4]:
# Cell 4 — Vanilla RNN and LSTM model classes

class SequenceClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes, cell_type="RNN", dropout=0.0):
        super().__init__()

        cell_type = cell_type.upper()
        recurrent_dropout = dropout if num_layers > 1 else 0.0

        if cell_type == "RNN":
            self.recurrent = nn.RNN(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                nonlinearity="tanh",
                dropout=recurrent_dropout
            )
        elif cell_type == "LSTM":
            self.recurrent = nn.LSTM(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=recurrent_dropout
            )
        else:
            raise ValueError("cell_type must be 'RNN' or 'LSTM'")

        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, padded_sequences, lengths):
        packed = pack_padded_sequence(
            padded_sequences,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        if isinstance(self.recurrent, nn.LSTM):
            packed_output, (h_n, c_n) = self.recurrent(packed)
            final_hidden = h_n[-1]
        else:
            packed_output, h_n = self.recurrent(packed)
            final_hidden = h_n[-1]

        logits = self.classifier(final_hidden)
        return logits


In [5]:
# Cell 5 — Training, convergence, evaluation, and artifact saving

def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for padded, lengths, labels, _ in loader:
        padded = padded.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(padded, lengths)
            loss = criterion(logits, labels)

            if is_training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += batch_size

    mean_loss = total_loss / max(total_samples, 1)
    accuracy = total_correct / max(total_samples, 1)

    return mean_loss, accuracy


@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    y_true, y_pred, sample_ids = [], [], []

    for padded, lengths, labels, ids in loader:
        padded = padded.to(DEVICE, non_blocking=True)
        logits = model(padded, lengths)
        predictions = logits.argmax(dim=1).cpu().numpy()

        y_true.extend(labels.numpy().tolist())
        y_pred.extend(predictions.tolist())
        sample_ids.extend(ids)

    return np.asarray(y_true), np.asarray(y_pred), sample_ids


def train_one_configuration(
    dataset_name,
    dataset_root,
    class_mapping,
    cell_type,
    config_name,
    hidden_dim,
    num_layers,
    dropout=0.0
):
    train_loader, train_eval_loader, val_loader, test_loader = make_loaders(
        dataset_root, class_mapping
    )

    input_dim = 2 if dataset_name == "handwriting" else 39
    num_classes = len(class_mapping)

    model = SequenceClassifier(
        input_dim=input_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        num_classes=num_classes,
        cell_type=cell_type,
        dropout=dropout
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    run_dir = WORK_DIR / dataset_name / cell_type.lower() / config_name
    run_dir.mkdir(parents=True, exist_ok=True)

    history = []
    previous_train_loss = None
    stable_epochs = 0
    converged = False

    print(f"\nStarting {dataset_name} | {cell_type} | {config_name}")
    print(model)

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, optimizer=None
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "validation_loss": val_loss,
            "validation_accuracy": val_acc
        })

        loss_delta = (
            abs(train_loss - previous_train_loss)
            if previous_train_loss is not None
            else float("inf")
        )

        if loss_delta < CONVERGENCE_THRESHOLD:
            stable_epochs += 1
        else:
            stable_epochs = 0

        if epoch%50==0:

            print(
                f"Epoch {epoch:03d} | train loss={train_loss:.6f} "
                f"| train acc={train_acc:.4f} | val loss={val_loss:.6f} "
                f"| val acc={val_acc:.4f} | Δloss={loss_delta:.8f}"
            )

        previous_train_loss = train_loss

        if stable_epochs >= CONVERGENCE_CONSECUTIVE_EPOCHS:
            converged = True
            print(
                f"Converged at epoch {epoch}: training loss difference "
                f"< {CONVERGENCE_THRESHOLD} for "
                f"{CONVERGENCE_CONSECUTIVE_EPOCHS} consecutive epochs."
            )
            break

    if not converged:
        print(
            f"Reached safety cap MAX_EPOCHS={MAX_EPOCHS} "
            "without satisfying convergence criterion."
        )

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / "training_history.csv", index=False)

    final_train_loss, final_train_acc = run_epoch(
        model, train_eval_loader, criterion, optimizer=None
    )
    final_val_loss, final_val_acc = run_epoch(
        model, val_loader, criterion, optimizer=None
    )
    final_test_loss, final_test_acc = run_epoch(
        model, test_loader, criterion, optimizer=None
    )

    y_true, y_pred, sample_ids = collect_predictions(model, test_loader)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(num_classes))
    )

    # Save checkpoint.
    torch.save({
        "model_state_dict": model.state_dict(),
        "cell_type": cell_type,
        "dataset_name": dataset_name,
        "config_name": config_name,
        "input_dim": input_dim,
        "hidden_dim": hidden_dim,
        "num_layers": num_layers,
        "dropout": dropout,
        "class_mapping": class_mapping,
        "converged": converged,
        "epochs_trained": len(history)
    }, run_dir / "model.pth")

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="Training loss")
    plt.plot(history_df["epoch"], history_df["validation_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Average Cross-Entropy Error")
    plt.title(f"{dataset_name} {cell_type} — {config_name}")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(run_dir / "loss_curve.png", dpi=200)
    plt.close()

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            name for name, idx in sorted(class_mapping.items(), key=lambda kv: kv[1])
        ]
    )
    fig, ax = plt.subplots(figsize=(7, 6))
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(f"Test Confusion Matrix — {dataset_name} {cell_type} {config_name}")
    plt.tight_layout()
    plt.savefig(run_dir / "confusion_matrix.png", dpi=200)
    plt.close(fig)

    # Save test predictions and report.
    pd.DataFrame({
        "sample_id": sample_ids,
        "true_label": y_true,
        "predicted_label": y_pred
    }).to_csv(run_dir / "test_predictions.csv", index=False)

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(num_classes)),
        target_names=[
            name for name, idx in sorted(class_mapping.items(), key=lambda kv: kv[1])
        ],
        zero_division=0
    )
    with open(run_dir / "classification_report.txt", "w") as f:
        f.write(report)

    result = {
        "dataset": dataset_name,
        "model": cell_type,
        "configuration": config_name,
        "hidden_dim": hidden_dim,
        "num_layers": num_layers,
        "dropout": dropout,
        "epochs": len(history),
        "converged": converged,
        "final_training_loss": final_train_loss,
        "training_accuracy": final_train_acc,
        "validation_accuracy": final_val_acc,
        "test_loss": final_test_loss,
        "test_accuracy": final_test_acc,
        "results_directory": str(run_dir)
    }

    with open(run_dir / "metrics.json", "w") as f:
        json.dump(result, f, indent=4)

    print("\nRESULT:", result)
    print("\nTest classification report:\n", report)

    return result


In [6]:
# Cell 6 — Architecture configurations (3 per model family, 12 total runs)

RNN_CONFIGS = [
    {"name": "RNN_1_1layer_32hidden", "hidden_dim": 32, "num_layers": 1, "dropout": 0.0},
    {"name": "RNN_2_2layers_64hidden", "hidden_dim": 64, "num_layers": 2, "dropout": 0.1},
    {"name": "RNN_3_3layers_128hidden", "hidden_dim": 128, "num_layers": 3, "dropout": 0.2},
]

LSTM_CONFIGS = [
    {"name": "LSTM_1_1layer_32hidden", "hidden_dim": 32, "num_layers": 1, "dropout": 0.0},
    {"name": "LSTM_2_2layers_64hidden", "hidden_dim": 64, "num_layers": 2, "dropout": 0.1},
    {"name": "LSTM_3_3layers_128hidden", "hidden_dim": 128, "num_layers": 3, "dropout": 0.2},
]

def run_configuration_set(dataset_name, dataset_root, class_mapping, cell_type, configs):
    results = []
    for config in configs:
        result = train_one_configuration(
            dataset_name=dataset_name,
            dataset_root=dataset_root,
            class_mapping=class_mapping,
            cell_type=cell_type,
            config_name=config["name"],
            hidden_dim=config["hidden_dim"],
            num_layers=config["num_layers"],
            dropout=config["dropout"]
        )
        results.append(result)

    results_df = pd.DataFrame(results)
    summary_path = WORK_DIR / dataset_name / f"{cell_type.lower()}_comparison.csv"
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(summary_path, index=False)

    print("\nCOMPARISON:", dataset_name, cell_type)
    display(results_df)

    return results_df


In [7]:
# Cell 7 — Four separate functions to call

def train_handwriting_rnn():
    return run_configuration_set(
        dataset_name="handwriting",
        dataset_root=HANDWRITING_ROOT,
        class_mapping=HANDWRITING_CLASS_MAPPING,
        cell_type="RNN",
        configs=RNN_CONFIGS
    )


def train_handwriting_lstm():
    return run_configuration_set(
        dataset_name="handwriting",
        dataset_root=HANDWRITING_ROOT,
        class_mapping=HANDWRITING_CLASS_MAPPING,
        cell_type="LSTM",
        configs=LSTM_CONFIGS
    )


def train_cv_rnn():
    return run_configuration_set(
        dataset_name="cv_speech",
        dataset_root=CV_ROOT,
        class_mapping=CV_CLASS_MAPPING,
        cell_type="RNN",
        configs=RNN_CONFIGS
    )


def train_cv_lstm():""
    return run_configuration_set(
        dataset_name="cv_speech",
        dataset_root=CV_ROOT,
        class_mapping=CV_CLASS_MAPPING,
        cell_type="LSTM",
        configs=LSTM_CONFIGS
    )

print("Functions defined. Each function runs 3 configurations.")


Functions defined. Each function runs 3 configurations.


In [8]:

handwriting_rnn_results = train_handwriting_rnn()



handwriting: total train=346, train split=276, validation=70, test=100
Train labels: Counter({np.int64(1): 56, np.int64(2): 56, np.int64(4): 55, np.int64(0): 55, np.int64(3): 54})
Validation labels: Counter({np.int64(3): 14, np.int64(4): 14, np.int64(0): 14, np.int64(2): 14, np.int64(1): 14})
Test labels: Counter({np.int64(0): 20, np.int64(1): 20, np.int64(2): 20, np.int64(3): 20, np.int64(4): 20})

Starting handwriting | RNN | RNN_1_1layer_32hidden
SequenceClassifier(
  (recurrent): RNN(2, 32, batch_first=True)
  (classifier): Linear(in_features=32, out_features=5, bias=True)
)
Epoch 050 | train loss=0.792839 | train acc=0.6159 | val loss=0.758222 | val acc=0.6000 | Δloss=0.01413503
Epoch 100 | train loss=0.681446 | train acc=0.6486 | val loss=0.640094 | val acc=0.6714 | Δloss=0.02032295
Epoch 150 | train loss=0.614849 | train acc=0.6957 | val loss=0.613508 | val acc=0.7000 | Δloss=0.00647672
Epoch 200 | train loss=0.603987 | train acc=0.6957 | val loss=0.624024 | val acc=0.7143 | Δlo

,dataset,model,configuration,hidden_dim,num_layers,dropout,epochs,converged,final_training_loss,training_accuracy,validation_accuracy,test_loss,test_accuracy,results_directory
0,handwriting,RNN,RNN_1_1layer_32hidden,32,1,0.0,1998,True,0.001477,1.0,0.885714,0.287838,0.95,/kaggle/working/assignment6_results/handwritin...
1,handwriting,RNN,RNN_2_2layers_64hidden,64,2,0.1,1905,True,0.003875,1.0,0.742857,0.607785,0.82,/kaggle/working/assignment6_results/handwritin...
2,handwriting,RNN,RNN_3_3layers_128hidden,128,3,0.2,920,True,0.000602,1.0,0.771429,0.907600,0.82,/kaggle/working/assignment6_results/handwritin...


In [9]:

handwriting_lstm_results = train_handwriting_lstm()


handwriting: total train=346, train split=276, validation=70, test=100
Train labels: Counter({np.int64(1): 56, np.int64(2): 56, np.int64(4): 55, np.int64(0): 55, np.int64(3): 54})
Validation labels: Counter({np.int64(3): 14, np.int64(4): 14, np.int64(0): 14, np.int64(2): 14, np.int64(1): 14})
Test labels: Counter({np.int64(0): 20, np.int64(1): 20, np.int64(2): 20, np.int64(3): 20, np.int64(4): 20})

Starting handwriting | LSTM | LSTM_1_1layer_32hidden
SequenceClassifier(
  (recurrent): LSTM(2, 32, batch_first=True)
  (classifier): Linear(in_features=32, out_features=5, bias=True)
)
Epoch 050 | train loss=0.772993 | train acc=0.5870 | val loss=0.724654 | val acc=0.6143 | Δloss=0.02661794
Epoch 100 | train loss=0.417295 | train acc=0.8297 | val loss=0.468665 | val acc=0.8286 | Δloss=0.02088364
Epoch 150 | train loss=0.236402 | train acc=0.9094 | val loss=0.468672 | val acc=0.8286 | Δloss=0.04252971
Epoch 200 | train loss=0.192895 | train acc=0.9312 | val loss=0.280011 | val acc=0.9000 | 

,dataset,model,configuration,hidden_dim,num_layers,dropout,epochs,converged,final_training_loss,training_accuracy,validation_accuracy,test_loss,test_accuracy,results_directory
0,handwriting,LSTM,LSTM_1_1layer_32hidden,32,1,0.0,540,True,0.002847,1.0,0.942857,0.203002,0.96,/kaggle/working/assignment6_results/handwritin...
1,handwriting,LSTM,LSTM_2_2layers_64hidden,64,2,0.1,194,True,0.001856,1.0,0.957143,0.001339,1.00,/kaggle/working/assignment6_results/handwritin...
2,handwriting,LSTM,LSTM_3_3layers_128hidden,128,3,0.2,164,True,0.001167,1.0,0.985714,0.000937,1.00,/kaggle/working/assignment6_results/handwritin...


In [10]:

cv_rnn_results = train_cv_rnn()


cv_speech: total train=1439, train split=1151, validation=288, test=360
Train labels: Counter({np.int64(1): 306, np.int64(3): 274, np.int64(2): 255, np.int64(0): 167, np.int64(4): 149})
Validation labels: Counter({np.int64(1): 77, np.int64(3): 69, np.int64(2): 63, np.int64(0): 42, np.int64(4): 37})
Test labels: Counter({np.int64(1): 96, np.int64(3): 86, np.int64(2): 80, np.int64(0): 52, np.int64(4): 46})

Starting cv_speech | RNN | RNN_1_1layer_32hidden
SequenceClassifier(
  (recurrent): RNN(39, 32, batch_first=True)
  (classifier): Linear(in_features=32, out_features=5, bias=True)
)
Epoch 050 | train loss=0.463735 | train acc=0.8271 | val loss=1.138008 | val acc=0.5208 | Δloss=0.00798446
Epoch 100 | train loss=0.223978 | train acc=0.9409 | val loss=1.557306 | val acc=0.5417 | Δloss=0.00406888
Epoch 150 | train loss=0.108670 | train acc=0.9791 | val loss=1.942325 | val acc=0.5521 | Δloss=0.00422862
Epoch 200 | train loss=0.063997 | train acc=0.9913 | val loss=2.368900 | val acc=0.5347 

,dataset,model,configuration,hidden_dim,num_layers,dropout,epochs,converged,final_training_loss,training_accuracy,validation_accuracy,test_loss,test_accuracy,results_directory
0,cv_speech,RNN,RNN_1_1layer_32hidden,32,1,0.0,386,True,0.005373,1.0,0.559028,2.839755,0.627778,/kaggle/working/assignment6_results/cv_speech/...
1,cv_speech,RNN,RNN_2_2layers_64hidden,64,2,0.1,497,True,0.000075,1.0,0.673611,2.587872,0.700000,/kaggle/working/assignment6_results/cv_speech/...
2,cv_speech,RNN,RNN_3_3layers_128hidden,128,3,0.2,312,True,0.000114,1.0,0.694444,2.007417,0.719444,/kaggle/working/assignment6_results/cv_speech/...


In [11]:

cv_lstm_results = train_cv_lstm()


cv_speech: total train=1439, train split=1151, validation=288, test=360
Train labels: Counter({np.int64(1): 306, np.int64(3): 274, np.int64(2): 255, np.int64(0): 167, np.int64(4): 149})
Validation labels: Counter({np.int64(1): 77, np.int64(3): 69, np.int64(2): 63, np.int64(0): 42, np.int64(4): 37})
Test labels: Counter({np.int64(1): 96, np.int64(3): 86, np.int64(2): 80, np.int64(0): 52, np.int64(4): 46})

Starting cv_speech | LSTM | LSTM_1_1layer_32hidden
SequenceClassifier(
  (recurrent): LSTM(39, 32, batch_first=True)
  (classifier): Linear(in_features=32, out_features=5, bias=True)
)
Epoch 050 | train loss=0.016307 | train acc=1.0000 | val loss=0.703031 | val acc=0.8299 | Δloss=0.00094372
Converged at epoch 93: training loss difference < 0.0001 for 3 consecutive epochs.

RESULT: {'dataset': 'cv_speech', 'model': 'LSTM', 'configuration': 'LSTM_1_1layer_32hidden', 'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.0, 'epochs': 93, 'converged': True, 'final_training_loss': 0.0027388020047

,dataset,model,configuration,hidden_dim,num_layers,dropout,epochs,converged,final_training_loss,training_accuracy,validation_accuracy,test_loss,test_accuracy,results_directory
0,cv_speech,LSTM,LSTM_1_1layer_32hidden,32,1,0.0,93,True,0.002739,1.0,0.836806,0.747185,0.847222,/kaggle/working/assignment6_results/cv_speech/...
1,cv_speech,LSTM,LSTM_2_2layers_64hidden,64,2,0.1,55,True,0.000567,1.0,0.850694,0.738721,0.872222,/kaggle/working/assignment6_results/cv_speech/...
2,cv_speech,LSTM,LSTM_3_3layers_128hidden,128,3,0.2,49,True,0.000386,1.0,0.878472,0.817921,0.869444,/kaggle/working/assignment6_results/cv_speech/...


In [12]:

all_results = pd.concat(
    [
        handwriting_rnn_results,
        handwriting_lstm_results,
        cv_rnn_results,
        cv_lstm_results
    ],
    ignore_index=True
)

combined_csv = WORK_DIR / "all_12_runs_comparison.csv"
all_results.to_csv(combined_csv, index=False)

display(
    all_results.sort_values(
        ["dataset", "test_accuracy"],
        ascending=[True, False]
    )
)

print("Combined results saved to:", combined_csv)
print("All artifacts saved under:", WORK_DIR)


,dataset,model,configuration,hidden_dim,num_layers,dropout,epochs,converged,final_training_loss,training_accuracy,validation_accuracy,test_loss,test_accuracy,results_directory
10,cv_speech,LSTM,LSTM_2_2layers_64hidden,64,2,0.1,55,True,0.000567,1.0,0.850694,0.738721,0.872222,/kaggle/working/assignment6_results/cv_speech/...
11,cv_speech,LSTM,LSTM_3_3layers_128hidden,128,3,0.2,49,True,0.000386,1.0,0.878472,0.817921,0.869444,/kaggle/working/assignment6_results/cv_speech/...
9,cv_speech,LSTM,LSTM_1_1layer_32hidden,32,1,0.0,93,True,0.002739,1.0,0.836806,0.747185,0.847222,/kaggle/working/assignment6_results/cv_speech/...
8,cv_speech,RNN,RNN_3_3layers_128hidden,128,3,0.2,312,True,0.000114,1.0,0.694444,2.007417,0.719444,/kaggle/working/assignment6_results/cv_speech/...
7,cv_speech,RNN,RNN_2_2layers_64hidden,64,2,0.1,497,True,0.000075,1.0,0.673611,2.587872,0.700000,/kaggle/working/assignment6_results/cv_speech/...
6,cv_speech,RNN,RNN_1_1layer_32hidden,32,1,0.0,386,True,0.005373,1.0,0.559028,2.839755,0.627778,/kaggle/working/assignment6_results/cv_speech/...
4,handwriting,LSTM,LSTM_2_2layers_64hidden,64,2,0.1,194,True,0.001856,1.0,0.957143,0.001339,1.000000,/kaggle/working/assignment6_results/handwritin...
5,handwriting,LSTM,LSTM_3_3layers_128hidden,128,3,0.2,164,True,0.001167,1.0,0.985714,0.000937,1.000000,/kaggle/working/assignment6_results/handwritin...
3,handwriting,LSTM,LSTM_1_1layer_32hidden,32,1,0.0,540,True,0.002847,1.0,0.942857,0.203002,0.960000,/kaggle/working/assignment6_results/handwritin...
0,handwriting,RNN,RNN_1_1layer_32hidden,32,1,0.0,1998,True,0.001477,1.0,0.885714,0.287838,0.950000,/kaggle/working/assignment6_results/handwritin...


Combined results saved to: /kaggle/working/assignment6_results/all_12_runs_comparison.csv
All artifacts saved under: /kaggle/working/assignment6_results


In [13]:
!zip -r "assignment6_results.zip" /kaggle/working/assignment6_results

  adding: kaggle/working/assignment6_results/ (stored 0%)
  adding: kaggle/working/assignment6_results/all_12_runs_comparison.csv (deflated 71%)
  adding: kaggle/working/assignment6_results/handwriting/ (stored 0%)
  adding: kaggle/working/assignment6_results/handwriting/lstm_comparison.csv (deflated 56%)
  adding: kaggle/working/assignment6_results/handwriting/rnn/ (stored 0%)
  adding: kaggle/working/assignment6_results/handwriting/rnn/RNN_1_1layer_32hidden/ (stored 0%)
  adding: kaggle/working/assignment6_results/handwriting/rnn/RNN_1_1layer_32hidden/loss_curve.png (deflated 4%)
  adding: kaggle/working/assignment6_results/handwriting/rnn/RNN_1_1layer_32hidden/metrics.json (deflated 47%)
  adding: kaggle/working/assignment6_results/handwriting/rnn/RNN_1_1layer_32hidden/test_predictions.csv (deflated 81%)
  adding: kaggle/working/assignment6_results/handwriting/rnn/RNN_1_1layer_32hidden/training_history.csv (deflated 69%)
  adding: kaggle/working/assignment6_results/handwriting/rnn/R